In [1]:
import pandas as pd
import numpy as np


RANDOM_STATE = 42

In [2]:
df_train_raw = pd.read_csv(f"data-00-raw/bike_train.csv", index_col=[0], parse_dates=["date"])

df_test_raw = pd.read_csv(f"data-00-raw/bike_test.csv", index_col=[0], parse_dates=["date"])

df_train_raw

,date,wcond,temp,atemp,hum,wind,count
1,2022-01-01,2,9.1,13.2,79.78,10.43,982
2,2022-01-02,2,9.9,12.7,68.91,16.16,797
3,2022-01-03,1,3.1,4.5,43.29,16.14,1351
4,2022-01-04,1,3.2,5.6,58.45,10.42,1559
5,2022-01-05,1,4.3,6.5,43.26,12.15,1597
...,...,...,...,...,...,...,...
361,2022-12-27,2,8.3,11.4,75.49,12.25,1161
362,2022-12-28,1,7.3,9.0,49.89,19.11,2301
363,2022-12-29,1,5.2,8.2,56.84,7.76,2421
364,2022-12-30,1,7.8,10.9,63.03,8.73,2995


In [3]:
from scripts.add_features import add_calendar_features_lgb, add_weather_features_lgb, get_lgb_features_cols

In [4]:
df_train = add_calendar_features_lgb(add_weather_features_lgb(df_train_raw))

df_test = add_calendar_features_lgb(add_weather_features_lgb(df_test_raw))

df_train

,date,wcond,temp,atemp,hum,wind,count,temp_sq,hum_sq,wind_sq,...,dayofweek,weekend,dayofyear,quarter,sin_doy_1,cos_doy_1,sin_doy_2,cos_doy_2,sin_doy_3,cos_doy_3
1,2022-01-01,2,9.1,13.2,79.78,10.43,982,82.81,6364.8484,108.7849,...,5,1,1,1,1.721336e-02,0.999852,3.442161e-02,0.999407,5.161967e-02,0.998667
2,2022-01-02,2,9.9,12.7,68.91,16.16,797,98.01,4748.5881,261.1456,...,6,1,2,1,3.442161e-02,0.999407,6.880243e-02,0.997630,1.031017e-01,0.994671
3,2022-01-03,1,3.1,4.5,43.29,16.14,1351,9.61,1874.0241,260.4996,...,0,0,3,1,5.161967e-02,0.998667,1.031017e-01,0.994671,1.543088e-01,0.988023
4,2022-01-04,1,3.2,5.6,58.45,10.42,1559,10.24,3416.4025,108.5764,...,1,0,4,1,6.880243e-02,0.997630,1.372788e-01,0.990532,2.051045e-01,0.978740
5,2022-01-05,1,4.3,6.5,43.26,12.15,1597,18.49,1871.4276,147.6225,...,2,0,5,1,8.596480e-02,0.996298,1.712931e-01,0.985220,2.553533e-01,0.966848
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2022-12-27,2,8.3,11.4,75.49,12.25,1161,68.89,5698.7401,150.0625,...,1,0,361,4,-6.880243e-02,0.997630,-1.372788e-01,0.990532,-2.051045e-01,0.978740
362,2022-12-28,1,7.3,9.0,49.89,19.11,2301,53.29,2489.0121,365.1921,...,2,0,362,4,-5.161967e-02,0.998667,-1.031017e-01,0.994671,-1.543088e-01,0.988023
363,2022-12-29,1,5.2,8.2,56.84,7.76,2421,27.04,3230.7856,60.2176,...,3,0,363,4,-3.442161e-02,0.999407,-6.880243e-02,0.997630,-1.031017e-01,0.994671
364,2022-12-30,1,7.8,10.9,63.03,8.73,2995,60.84,3972.7809,76.2129,...,4,0,364,4,-1.721336e-02,0.999852,-3.442161e-02,0.999407,-5.161967e-02,0.998667


# Cross Validation

### Preparing Data

In [9]:
y = df_train["count"].to_numpy()

In [10]:
df_train_X = df_train[get_lgb_features_cols()].copy()

df_test_X = df_test[get_lgb_features_cols()].copy()

df_train_X

,temp,atemp,hum,wind,wcond,temp_sq,hum_sq,wind_sq,temp_x_hum,temp_x_wind,...,dayofweek,weekend,dayofyear,quarter,sin_doy_1,cos_doy_1,sin_doy_2,cos_doy_2,sin_doy_3,cos_doy_3
1,9.1,13.2,79.78,10.43,2,82.81,6364.8484,108.7849,725.998,94.913,...,5,1,1,1,1.721336e-02,0.999852,3.442161e-02,0.999407,5.161967e-02,0.998667
2,9.9,12.7,68.91,16.16,2,98.01,4748.5881,261.1456,682.209,159.984,...,6,1,2,1,3.442161e-02,0.999407,6.880243e-02,0.997630,1.031017e-01,0.994671
3,3.1,4.5,43.29,16.14,1,9.61,1874.0241,260.4996,134.199,50.034,...,0,0,3,1,5.161967e-02,0.998667,1.031017e-01,0.994671,1.543088e-01,0.988023
4,3.2,5.6,58.45,10.42,1,10.24,3416.4025,108.5764,187.040,33.344,...,1,0,4,1,6.880243e-02,0.997630,1.372788e-01,0.990532,2.051045e-01,0.978740
5,4.3,6.5,43.26,12.15,1,18.49,1871.4276,147.6225,186.018,52.245,...,2,0,5,1,8.596480e-02,0.996298,1.712931e-01,0.985220,2.553533e-01,0.966848
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,8.3,11.4,75.49,12.25,2,68.89,5698.7401,150.0625,626.567,101.675,...,1,0,361,4,-6.880243e-02,0.997630,-1.372788e-01,0.990532,-2.051045e-01,0.978740
362,7.3,9.0,49.89,19.11,1,53.29,2489.0121,365.1921,364.197,139.503,...,2,0,362,4,-5.161967e-02,0.998667,-1.031017e-01,0.994671,-1.543088e-01,0.988023
363,5.2,8.2,56.84,7.76,1,27.04,3230.7856,60.2176,295.568,40.352,...,3,0,363,4,-3.442161e-02,0.999407,-6.880243e-02,0.997630,-1.031017e-01,0.994671
364,7.8,10.9,63.03,8.73,1,60.84,3972.7809,76.2129,491.634,68.094,...,4,0,364,4,-1.721336e-02,0.999852,-3.442161e-02,0.999407,-5.161967e-02,0.998667


### Selecting Parameters

In [15]:
from sklearn.model_selection import KFold
import optuna
from scripts.find_solve_lightgbm import make_objective, fit_final_lgbm

In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

N_TRIALS = 100

study = optuna.create_study(direction="minimize")

study.optimize(make_objective(df_test_X, y, kf), n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
best_mse = study.best_value
best_rmse = np.sqrt(best_mse)

[I 2026-06-01 14:04:04,229] A new study created in memory with name: no-name-1dbf4058-a551-4f5b-8840-440b80e50137


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-06-01 14:04:04,348] Trial 0 finished with value: 551947.7897685936 and parameters: {'n_estimators': 551, 'learning_rate': 0.013742977871169595, 'num_leaves': 104, 'max_depth': 9, 'min_child_samples': 46, 'subsample': 0.7011904828235671, 'colsample_bytree': 0.8587269509691378, 'reg_alpha': 2.3529315604724275, 'reg_lambda': 0.018418881227566488}. Best is trial 0 with value: 551947.7897685936.
[I 2026-06-01 14:04:04,679] Trial 1 finished with value: 667556.1935041352 and parameters: {'n_estimators': 895, 'learning_rate': 0.0014544742378309878, 'num_leaves': 82, 'max_depth': 5, 'min_child_samples': 8, 'subsample': 0.6374527169776074, 'colsample_bytree': 0.6963006792232392, 'reg_alpha': 0.021971201890585604, 'reg_lambda': 5.921092131773981}. Best is trial 0 with value: 551947.7897685936.
[I 2026-06-01 14:04:04,714] Trial 2 finished with value: 554068.3337419977 and parameters: {'n_estimators': 487, 'learning_rate': 0.13464909074483206, 'num_leaves': 46, 'max_depth': 4, 'min_child_sa

In [14]:
print(f"Best CV MSE  : {best_mse:.2f}")
print(f"Best CV RMSE : {best_rmse:.2f}")
print(f"Best params  : {best_params}")

Best CV MSE  : 514771.31
Best CV RMSE : 717.48
Best params  : {'n_estimators': 701, 'learning_rate': 0.23796774757282269, 'num_leaves': 59, 'max_depth': 3, 'min_child_samples': 27, 'subsample': 0.8224198027488112, 'colsample_bytree': 0.8529719414772412, 'reg_alpha': 4.722619165820532, 'reg_lambda': 0.00010300875864259597}


In [16]:
model = fit_final_lgbm(df_test_X, y, best_params)

In [18]:
importance = pd.Series(
    model.feature_importances_, index=get_lgb_features_cols()
).sort_values(ascending=False)
print("\nFeature importances (gain):")
print(importance.head(10).round(2))


Feature importances (gain):
temp_x_wind        332
sin_doy_3          281
hum                270
temp_atemp_diff    218
temp_x_hum         205
sin_doy_2          204
wind               192
cos_doy_2          172
cos_doy_3          168
temp               138
dtype: int32


# Save Submission

In [24]:
from scripts.save_submission import save_submission_csv

In [25]:
preds = model.predict(df_test_X)

df_submission = pd.DataFrame({"date": df_test_raw["date"], "pred": preds})

df_submission

,date,pred
1,2023-01-01,1072.637784
2,2023-01-02,826.803604
3,2023-01-03,1350.060570
4,2023-01-04,1542.820781
5,2023-01-05,1599.821451
...,...,...
361,2023-12-27,1207.326123
362,2023-12-28,2265.547368
363,2023-12-29,2437.864664
364,2023-12-30,2952.667331


In [26]:
save_submission_csv(df_submission)